# Prompt Optimization

**Module:** 07-prompt-engineering

**Notebook:** `08-prompt-optimization.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Prompt Testing** with clear contracts and failure modes
- Explain and apply **Versioning** with clear contracts and failure modes
- Explain and apply **A/B Testing** with clear contracts and failure modes
- Explain and apply **Token & Cost Optimization** with clear contracts and failure modes
- Explain and apply **Performance Evaluation** with clear contracts and failure modes
- Explain and apply **Optimization Loop** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Prompt Optimization

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Prompt Testing**
2. **Versioning**
3. **A/B Testing**
4. **Token & Cost Optimization**
5. **Performance Evaluation**
6. **Optimization Loop**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Prompt Testing

### Definition
**Prompt Testing** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Prompt Testing typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Prompt Testing: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Prompt Testing as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Prompt Testing as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Prompt Testing
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Prompt Testing when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Prompt Testing improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Prompt Testing" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Prompt Testing"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


## Versioning

### Definition
**Versioning** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Versioning typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Versioning: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Versioning as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Versioning as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Versioning
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Versioning when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Versioning" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Versioning"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Versioning

**Situation:** A team wants to productionize a feature involving **Versioning**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## A/B Testing

### Definition
**A/B Testing** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around A/B Testing typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For A/B Testing: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain A/B Testing as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating A/B Testing as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for A/B Testing
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use A/B Testing when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "A/B Testing" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "A/B Testing"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


## Token & Cost Optimization

### Definition
**Token & Cost Optimization** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Token & Cost Optimization typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Token & Cost Optimization: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Token & Cost Optimization as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Token & Cost Optimization as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Token & Cost Optimization
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Token & Cost Optimization when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Token & Cost Optimization" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Token & Cost Optimization"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def usage_account(prompt: str, completion: str, price_in=0.15, price_out=0.60):
    # prices are illustrative $/1M tokens
    tin, tout = approx_tokens(prompt), approx_tokens(completion)
    cost = (tin * price_in + tout * price_out) / 1_000_000
    return {"prompt_tokens": tin, "completion_tokens": tout, "usd_estimate": round(cost, 6)}

print(usage_account("system+user..." * 50, "answer..." * 20))


In [ ]:
# Streaming chunk assembler (shape similar to provider events)
chunks = [{"delta": "Hello"}, {"delta": ", "}, {"delta": "world"}]
out = []
for ch in chunks:
    out.append(ch["delta"])
    print("partial:", "".join(out))
print("final:", "".join(out))


### Worked scenario — Token & Cost Optimization

**Situation:** A team wants to productionize a feature involving **Token & Cost Optimization**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Performance Evaluation

### Definition
**Performance Evaluation** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Performance Evaluation typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Performance Evaluation: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Performance Evaluation as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Performance Evaluation as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Performance Evaluation
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Performance Evaluation when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Performance Evaluation" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Performance Evaluation"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


## Optimization Loop

### Definition
**Optimization Loop** is a core building block in 08-prompt-optimization within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Optimization Loop typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Optimization Loop: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Optimization Loop as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Optimization Loop as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Optimization Loop
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Optimization Loop when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Optimization Loop" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Optimization Loop"
    notebook: str = "08-prompt-optimization"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Optimization Loop

**Situation:** A team wants to productionize a feature involving **Optimization Loop**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Prompt Optimization**.

| Topic | Do | Don't |
|-------|----|-------|
| Prompt Testing | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Versioning | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| A/B Testing | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Token & Cost Optimization | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Performance Evaluation | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Optimization Loop | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Prompt Testing | Key concept covered in this notebook; see its section for definition and pitfalls |
| Versioning | Key concept covered in this notebook; see its section for definition and pitfalls |
| A/B Testing | Key concept covered in this notebook; see its section for definition and pitfalls |
| Token & Cost Optimization | Key concept covered in this notebook; see its section for definition and pitfalls |
| Performance Evaluation | Key concept covered in this notebook; see its section for definition and pitfalls |
| Optimization Loop | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Prompt Optimization** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **Prompt Testing**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Versioning**, then fix your demo until it passes.
3. Implement a failing test/fixture for **A/B Testing**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Token & Cost Optimization**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Performance Evaluation**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
